In [ ]:
import sys
import importlib
import pandas as pd

from pathlib import Path
from IPython.display import display

# 프로젝트 루트 검색
current = Path.cwd().resolve()

ROOT_DIR = next(
    (
        path for path in [current, *current.parents]
        if (path / "src" / "cwru_preprocessing.py").is_file()
    ),
    None
)

if ROOT_DIR is None:
    raise FileNotFoundError("GummyBearing 프로젝트 경로를 확인하세요.")

sys.path.insert(0, str(ROOT_DIR))

# 두 모듈 모두 새로고침
import src.cwru_feature as cf
import src.cwru_preprocessing as cp

importlib.reload(cf)
importlib.reload(cp)

# 최신 함수 가져오기
extract_metadata = cf.extract_metadata
process_cwru_file = cp.process_cwru_file

print("프로젝트 경로:", ROOT_DIR)
print("함수 import 완료!")

In [ ]:
folder = ROOT_DIR / "DATA" / "CWRU_Bearing_Dataset" 

target_folders = ["fault", "normal"]

all_files = []

for folder_name in target_folders:

    folder_path = folder / folder_name

    if not folder_path.is_dir():
        raise FileNotFoundError(f"폴더 없음: {folder_path}")

    files = sorted(
        path for path in folder_path.rglob("*.csv")
        if path.is_file()
    )

    print(f"{folder_name}: {len(files)}개")

    all_files.extend(files)

print("전체 파일 수:", len(all_files))

In [ ]:
file_path = all_files[54]

result = process_cwru_file(file_path)

display(pd.DataFrame([result]))

In [ ]:
results = []
errors = []

for i, file_path in enumerate(all_files, start=1):

    try:
        # CSV 파일 → 특징값 한 행
        result = process_cwru_file(file_path)

        # 폴더명과 라벨 일치 여부 확인
        expected_label = (
            0 if file_path.relative_to(folder).parts[0] == "normal"
            else 1
        )

        if result["label"] != expected_label:
            raise ValueError("폴더명과 파일명의 정상/고장 정보가 다릅니다.")

        results.append(result)

    except Exception as e:

        errors.append({
            "file_path": str(file_path),
            "error": str(e)
        })

    if i % 50 == 0 or i == len(all_files):
        print(f"{i}/{len(all_files)}개 처리 완료")


# DataFrame 생성
cwru_df = pd.DataFrame(results)

error_df = pd.DataFrame(
    errors,
    columns=["file_path", "error"]
)

In [ ]:
print("========== CWRU 통합 결과 ==========")

print("전체 파일 수:", len(all_files))
print("처리 성공:", len(cwru_df))
print("처리 실패:", len(error_df))

display(cwru_df.head())

print("\n고장 종류별 파일 수:")
display(cwru_df["fault_type"].value_counts())

print("\n컬럼별 결측치:")
display(cwru_df.isnull().sum())

print("\n처리 실패 파일:")
display(error_df)
print(error_df["file_path"])

In [ ]:
columns = [
    "file_id",
    "fault_type",
    "fault_diameter",
    "load_hp",
    "rpm",
    "or_pos",
    "label",

    "DE_RMS",
    "DE_Kurtosis",
    "DE_Peak",
    "DE_Crest_Factor",
]

cwru_df = cwru_df.reindex(columns=columns)

display(cwru_df.head())

In [ ]:
len(cwru_df)

In [ ]:
output_dir = ROOT_DIR / "DATA" / "features"

output_dir.mkdir(parents=True, exist_ok=True)

cwru_df.to_csv(
    output_dir / "cwru_features.csv",
    index=False
)

error_df.to_csv(
    output_dir / "cwru_errors.csv",
    index=False
)

print("저장 완료!")